In [ ]:
rm(list=ls())
library(Seurat)
library(reticulate)
library(SeuratDisk)
library(dplyr)
library(MAST)

use_python("/home/cporter/ENTER/envs/scanpy/bin/python", required = TRUE)
print(py_config())

In [ ]:
packageVersion("Seurat")

In [ ]:
# make Seurat object from anndata 
library(sceasy)
sceasy::convertFormat('/home/EOCRC_atlas/data/all_samples_raw_withTier2Annotation.h5ad', from="anndata", to="seurat",
                       outFile='/home/EOCRC_atlas/data/all_samples_raw_withTier2Annotation_09-19-25.rds')

In [ ]:
# Jump in point 
crc <- readRDS('home/EOCRC_atlas/data/all_samples_raw_withTier2Annotation_09-19-25.rds')

In [ ]:
# normalize seurat object 
crc <- NormalizeData(crc, normalization.method = "LogNormalize", scale.factor = 10000)

In [ ]:
# save normalized object 
saveRDS(crc, '/home/cporter/atlas_remake_June_20_2025/data/all_samples_raw_withTier2Annotation_normalized_09-19-25.rds')

In [ ]:
# run only the subsets that aren't annotated as "mixed" 
run_celltypes = c('Adipocytes', 'B cell', 'CD4 T cells', 'CD8 T cells',
       'CEACAM1 colonocyte-like', 'Cycing endothelium', 'Cycling Myeloid',
       'Cycling Stromal', 'Cycling T cells', 'Cycling plasma cell', 'DC',
       'Enteroendocrine-like', 'Fibroblast', 'Fibroblast-BMP5-SOX6',
       'Fibroblast-C3', 'Fibroblast-Infl', 'Fibroblast-KCNN3',
       'Fibroblast-MMP2-THY1', 'Germinal center / Cycling B cell',
       'Glial cells', 'HSP-hi - B cell', 'HSP-hi Myeloid',
       'HSP-hi Stromal', 'HSP-hi T cells', 'HSP-hi glial', 'ILCs',
       'LGR5 stem cell-like', 'Lymphatic endothelium',
       'MT-Ribo-hi Myeloid', 'MT-Ribo-hi Stromal', 'MT-Ribo-hi T cells',
       'MT-Ribo-hi endothelium', 'MT-Ribo-hi epithelial',
       'MUC2 goblet-like', 'Macrophage-Monocyte', 'Mast',
       'Myofibroblast-SMC', 'NK-Cytotoxic T cells', 'Neuronal cells',
       'Neutrophil', 'Patient-specific', 'Pericytes', 'Plasma cell',
       'Regulatory T cells', 'T helper cells', 'Vascular endothelium')

In [ ]:
# create save path 
data.path = '/home/EOCRC_atlas/data/'
save.path = '/home/EOCRC_atlas/results/YOCRC_pctExpressed/'
if (!dir.exists(save.path)) {dir.create(save.path, recursive = TRUE)}
date = "DATE"

In [ ]:
# jump in point 
crc = readRDS(paste0(data.path, 'all_samples_raw_withTier2Annotation_normalized_09-19-25.rds'))

In [ ]:
# subset to desired cell types 
crc <- subset(crc, subset = Annotation_Tier2 %in% run_celltypes)

In [ ]:
unique(crc@meta.data$Annotation_Tier2) # check cell types 

In [ ]:
# split object by granular annotation 
groups <- unique(crc$Annotation_Tier2)
split_list <- list()

for (g in groups) {
  message("Splitting group: ", g)
  split_list[[g]] <- subset(crc, subset = Annotation_Tier2 == g)
  gc() 
}

In [ ]:
# make space 
rm(crc)

In [ ]:
# Create CSV files showing percent expressed for each gene in each cell type 

library(tidyr)

for (celltype in run_celltypes){ 
    tmp = split_list[[celltype]]
    data = GetAssayData(tmp, assay = "RNA", slot = "data")
    thresh.min <- 0
    features <- rownames(tmp)
    
    cells.1 = rownames(tmp@meta.data)[tmp@meta.data$MSI_v2 == 'MSS: STABLE' & tmp@meta.data$Cohort == 'UnderFifty']
    pct.1 <- round(rowSums(data[features, cells.1, drop = FALSE] > thresh.min) / length(x = cells.1), digits = 3)
    
    cells.2 = rownames(tmp@meta.data)[tmp@meta.data$MSI_v2=='MSI-H: HIGH' & tmp@meta.data$Cohort == 'UnderFifty']
    pct.2 <- round(rowSums(data[features, cells.2, drop = FALSE] > thresh.min) / length(x = cells.2), digits = 3)

    cells.3 = rownames(tmp@meta.data)[tmp@meta.data$MSI_v2=='MSS: STABLE' & tmp@meta.data$Cohort == 'FiftyPlus']
    pct.3 <- round(rowSums(data[features, cells.3, drop = FALSE] > thresh.min) / length(x = cells.3), digits = 3)
    
    cells.4 = rownames(tmp@meta.data)[tmp@meta.data$MSI_v2=='MSI-H: HIGH' & tmp@meta.data$Cohort == 'FiftyPlus']
    pct.4 <- round(rowSums(data[features, cells.4, drop = FALSE] > thresh.min) / length(x = cells.4), digits = 3)

    celltype_tmp = gsub('/', '_', celltype)

    df = data.frame("MSS_UnderFifty" = pct.1, "MSI_UnderFifty" = pct.2, "MSS_FiftyPlus" = pct.3, "MSI_FiftyPlus" = pct.4, "Gene"=features) 
    write.csv(df, paste0(save.path, date, '_pctExpressedGenes_Tier2_splitByMSICohort', celltype_tmp, '.csv'))
}